# 主要是測試各種功能用的，自行增加區塊來測試(記得加註解)

## torch測試及可用顯存測試

In [ ]:
import torch
from psutil import virtual_memory
print(torch.cuda.is_available())
GPUmem = torch.cuda.mem_get_info()
mem = GPUmem[1] /1024 / 1024 / 1024
mem += virtual_memory().total /1024 / 1024 / 1024 / 2
mem

## 下載YouTube影片及聲音測試

In [ ]:
from bs4 import BeautifulSoup
from pytubefix import YouTube
yt = YouTube('https://music.youtube.com/watch?v=NycFr6D6DSw&si=przy_9oxIVQs5HI9')
print('download...')
yt.streams.get_audio_only().download(output_path='download/')   #下載音訊
print('ok!')

In [ ]:
acc = yt.streams.get_audio_only()
acc.download(output_path='download/')   #下載音訊

### 加入手動輸入連結

In [ ]:
url = input()

In [ ]:
from YouTubeDownload import YouTubeDownload
from MusicTools import MusicTools
import os
vedio = YouTubeDownload.download_youtube_video(url)
audio = YouTubeDownload.download_youtube_audio(url)
separation_audio = MusicTools.run_separation(audio)
merge = YouTubeDownload.merge_video_audio(vedio, separation_audio[0])
print(vedio, audio , merge, separation_audio)
#os.system(merge)
separation_audio

In [ ]:
os.system(merge)
audio

### 整合ASR辨識

In [ ]:
ASR_result = MusicTools.run_ASR(separation_audio[1])

In [ ]:
YouTubeDownload.download_captions_by_language_code(video_url=url,language_code=ASR_result[1])

In [ ]:
from YouTubeDownload import YouTubeDownload
YouTubeDownload.download_captions_by_language_code(video_url=url,language_code='ja')

## Whisper字幕辨識測試

In [ ]:
import whisper
whisper.available_models()

In [ ]:
import os
import torch
import tkinter as tk
from tkinter import filedialog
import whisper

#建立Tk主視窗，但隱藏
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)  #對話框出現在最前方

#選擇音訊檔案
current_dir = os.getcwd()
file_path = filedialog.askopenfilename(
    title="請選擇音訊檔案",
    initialdir=current_dir,
    filetypes=[("音訊檔案", "*.wav *.mp3 *.m4a *.flac"), ("所有檔案", "*.*")]
)

if not file_path:
    print("未選擇檔案")
    exit()

print("選擇的檔案:", file_path)

#載入Whisper模型
if(torch.cuda.is_available()):
    model = whisper.load_model("large-v3", device="cuda")
else:
    model = whisper.load_model("large-v3", device="cpu")

#轉錄音訊，Whisper會自動偵測語言
result = model.transcribe(file_path)
print("偵測語言:", result["language"])

#輸出資料夾:與音訊檔同一層
audio_dir = os.path.dirname(file_path)

#取得音訊檔名稱(不含副檔名)
filename = os.path.splitext(os.path.basename(file_path))[0]

#---SRT---
with open(os.path.join(audio_dir, f"{filename}.srt"), "w", encoding="utf-8") as f:
    for i, seg in enumerate(result["segments"], start=1):
        text = seg['text'].strip()
        if not text:
            continue
        f.write(f"{i}\n")
        start = seg["start"]
        end = seg["end"]
        start_time = f"{int(start//3600):02}:{int((start%3600)//60):02}:{int(start%60):02},{int((start%1)*1000):03}"
        end_time = f"{int(end//3600):02}:{int((end%3600)//60):02}:{int(end%60):02},{int((end%1)*1000):03}"
        f.write(f"{start_time} --> {end_time}\n")
        f.write(f"{text}\n\n")

#---VTT---
with open(os.path.join(audio_dir, f"{filename}.vtt"), "w", encoding="utf-8") as f:
    f.write("WEBVTT\n\n")
    for seg in result["segments"]:
        text = seg['text'].strip()
        if not text:
            continue
        start = seg["start"]
        end = seg["end"]
        start_time = f"{int(start//3600):02}:{int((start%3600)//60):02}:{int(start%60):02}.{int((start%1)*1000):03}"
        end_time = f"{int(end//3600):02}:{int((end%3600)//60):02}:{int(end%60):02}.{int((end%1)*1000):03}"
        f.write(f"{start_time} --> {end_time}\n")
        f.write(f"{text}\n\n")

#---TXT---
with open(os.path.join(audio_dir, f"{filename}.txt"), "w", encoding="utf-8") as f:
    for seg in result["segments"]:
        text = seg['text'].strip()
        if not text:
            continue
        start = seg["start"]
        end = seg["end"]
        start_time = f"{int(start//3600):02}:{int((start%3600)//60):02}:{int(start%60):02}.{int((start%1)*1000):03}"
        end_time = f"{int(end//3600):02}:{int((end%3600)//60):02}:{int(end%60):02}.{int((end%1)*1000):03}"
        f.write(f"[{start_time} --> {end_time}] {text}\n")

print(f"已輸出至資料夾:{audio_dir}")


### 分離功能至MusicTools.py內測試

In [ ]:
from MusicTools import MusicTools
#MusicTools.run_ASR(r"H:/NFU/score/score/output/Immortals/Immortals-vocals.wav")    #若無傳入值則由使用者手動選取
MusicTools.run_ASR()

## Whisper微調測試(官方範例)

In [ ]:
from datasets import load_dataset, DatasetDict
import pandas as pd

common_voice = DatasetDict()

#載入文件範例("mozilla-foundation/common_voice_11_0"是線上的資料，要自己改)
common_voice["train"] = load_dataset("mozilla-foundation/common_voice_11_0", "hi", split="train+validation", trust_remote_code=True)
common_voice["test"] = load_dataset("mozilla-foundation/common_voice_11_0", "hi", split="test", trust_remote_code=True)

#common_voice = pd.read_csv("processed_songs/metadata_all.csv")
common_voice["train"]["audio"]

### 這裡開始

In [1]:
import pandas as pd
from datasets import load_dataset, Audio ,DatasetDict

#模型變數設置
#medium 
useModel = "openai/whisper-small"
language1 = "English" #"Chinese"
language = "en" #"zh"

#讀取原始CSV
df = pd.read_csv("processed_songs/metadata_all.csv")

df['file'] = "processed_songs/" + df['file'].astype(str)
#改欄位名稱符合Hugging Face標準
df = df.rename(columns={"file": "audio", "text": "sentence"})
df = df[["audio","sentence"]]

#存成新的CSV
df.to_csv("processed_songs/metadata_hf.csv", index=False)

#用Hugging Face載入新的CSV
dataset = load_dataset(
    "csv",
    data_files={"train": "processed_songs/metadata_hf.csv"}
)

#直接在cast_column中指定目標採樣率
#這會告訴datasets在讀取音檔時，自動解碼並重採樣到16000Hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
common_voice = dataset

#---開始偵錯---

#1.先把第一筆音訊資料取出來，存成一個變數
first_audio_data = dataset["train"][0]["audio"]

#2.印出這個變數的「型別」，看看它到底是什麼
print("變數的型別:", type(first_audio_data))
print("-" * 20)

#3.嘗試存取它裡面的key(例如"sampling_rate"或"array")
#這個動作會「強制」datasets 去執行解碼和重採樣
print("音訊的採樣率:", first_audio_data["sampling_rate"])
print("音訊波形陣列 (前5個點):", first_audio_data["array"][:5])
print("-" * 20)

#4.再次印出整個變數
print("強制解碼後，完整的變數內容:")
print(first_audio_data)

Generating train split: 0 examples [00:00, ? examples/s]

變數的型別: <class 'datasets.features._torchcodec.AudioDecoder'>
--------------------
音訊的採樣率: 16000
音訊波形陣列 (前5個點): [0. 0. 0. 0. 0.]
--------------------
強制解碼後，完整的變數內容:


In [2]:
#特徵提取器(預處理音檔)_選擇模型
#模型輸出後處理為文字格式的分詞器
#language要自己改
from transformers import WhisperFeatureExtractor, WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained(useModel)
tokenizer = WhisperTokenizer.from_pretrained(useModel, language=language1, task="transcribe")


In [3]:
input_str = common_voice["train"][0]["sentence"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")

Input:                 As long as you love me
Decoded w/ special:    <|startoftranscript|><|en|><|transcribe|><|notimestamps|>As long as you love me<|endoftext|>
Decoded w/out special: As long as you love me
Are equal:             True


In [4]:
from transformers import WhisperProcessor
"""輸出格式範例(採樣率48K)
{'audio': {'path': '/home/sanchit_huggingface_co/.cache/huggingface/datasets/downloads/extracted/607848c7e74a89a3b5225c0fa5ffb9470e39b7f11112db614962076a847f3abf/cv-corpus-11.0-2022-09-21/hi/clips/common_voice_hi_25998259.mp3', 
           'array': array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 9.6724887e-07,
       1.5334779e-06, 1.0415988e-06], dtype=float32), 
           'sampling_rate': 48000},
 'sentence': 'खीर की मिठास पर गरमाई बिहार की सियासत, कुशवाहा ने दी सफाई'}
"""
processor = WhisperProcessor.from_pretrained(useModel, language=language, task="transcribe")

#print(common_voice["train"][0])

In [5]:
#from datasets import Audio
"""輸出格式範例(下採樣至16K)
{'audio': {'path': '/home/sanchit_huggingface_co/.cache/huggingface/datasets/downloads/extracted/607848c7e74a89a3b5225c0fa5ffb9470e39b7f11112db614962076a847f3abf/cv-corpus-11.0-2022-09-21/hi/clips/common_voice_hi_25998259.mp3', 
           'array': array([ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
       -3.4206650e-07,  3.2979898e-07,  1.0042874e-06], dtype=float32),
           'sampling_rate': 16000},
 'sentence': 'खीर की मिठास पर गरमाई बिहार की सियासत, कुशवाहा ने दी सफाई'}
"""
#common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

print(common_voice["train"][0])

{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x0000020119B2B9D0>, 'sentence': 'As long as you love me'}


In [6]:
#初始化processor(包含feature_extractor + tokenizer)
import torch
def prepare_dataset(batch, processor, torch):
    #processor = WhisperProcessor.from_pretrained(useModel, language="zh", task="transcribe")
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    batch["labels"] = torch.tensor(batch["labels"], dtype=torch.long)
    return batch

common_voice = common_voice.map(
    prepare_dataset,
    fn_kwargs={"processor":processor,"torch":torch},
    remove_columns=common_voice["train"].column_names,
    num_proc=14  #取決於你的CPU核心，自己填數字
)


Map (num_proc=14):   0%|          | 0/1561 [00:00<?, ? examples/s]

In [7]:
#定義數據收集器
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        #split inputs and labels since they have to be of different lengths and need different padding methods
        #first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        #get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        #pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        #replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        #if bos token is appended in previous tokenization step,
        #cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [8]:
#初始化數據整理器
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.bos_token_id,  #Whisper的decoder開始token
)

In [ ]:
import evaluate

#wer代表詞錯誤率
metric = evaluate.load("wer")

#錯誤率評估函式
if language == "en":
    def compute_metrics(pred):
        #取出預測結果
        pred_ids = pred.predictions
        label_ids = pred.label_ids

        #如果predictions是tuple(logits,...)
        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]

        #如果是logits，取argmax
        if pred_ids.ndim == 3:
            pred_ids = pred_ids.argmax(-1)

        #把-100換成pad_token_id
        import numpy as np
        label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

        #解碼成文字
        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        pred_str = [s.lower() for s in pred_str]
        label_str = [s.lower() for s in label_str]

        #計算WER
        wer = 100 * metric.compute(predictions=pred_str, references=label_str)
        return {"wer": wer}
#中文版
elif language == "zh":
    import numpy as np
    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids

        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]

        if pred_ids.ndim == 3:
            pred_ids = pred_ids.argmax(-1)

        label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        pred_str = [" ".join(list(s.replace(" ", ""))) for s in pred_str]
        label_str = [" ".join(list(s.replace(" ", ""))) for s in label_str]

        wer_score = 100 * metric.compute(predictions=pred_str, references=label_str)

        return {"wer": wer_score}


In [10]:
#載入預訓練checkpoint
from transformers import WhisperForConditionalGeneration

#模型自己改
model = WhisperForConditionalGeneration.from_pretrained(useModel)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [11]:
from transformers import Seq2SeqTrainingArguments,EarlyStoppingCallback

#如果不想將模型checkpoint上傳到Hub，設定push_to_hub=False
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-en",
    per_device_train_batch_size=4,  #如果出現OOM(顯存超載)錯誤將此減半(初始值8)
    gradient_accumulation_steps=4,  #並將此乘2(初始值2)
    learning_rate=1e-5,
    warmup_steps=200,   # 縮短warmup，因為模型很快就收斂了
    max_steps=10000,
    gradient_checkpointing=True, #開啟這個可以節省顯存，雖然會慢一點
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    
    # --- 關鍵改動處 ---
    save_steps=100,            #每100步就檢查一次
    eval_steps=100,            #每100步就評估一次
    save_total_limit=2,        #只保留最好的兩個模型，省硬碟空間
    weight_decay=0.05,         #增加權重衰減，防止過擬合
    label_smoothing_factor=0.1, #增加標籤平滑，提高泛化能力
    # ----------------
    
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5 # 連續 3 次評估(3000 steps)沒有改善則停止
)

In [12]:
#從訓練資料集中抓出測試資料
common_voice = common_voice["train"].train_test_split(test_size=0.1)

In [13]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback],
)

C:\Users\a0987\AppData\Local\Temp\ipykernel_14844\1168595547.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [14]:
#訓練模型
trainer.train()

You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


ValueError: You have to specify either decoder_input_ids or decoder_inputs_embeds

In [ ]:
#設定上傳模型數值
kwargs = {
    "dataset_tags": "Zzzkay1/song", #標籤，記得改
    "dataset": "Chinese songs * 14",  #a 'pretty' name for the training dataset (翻譯)給資料集設定一個你喜歡的名子
    "dataset_args": "config: en, split: test",
    "language": "en", #語言自己改
    "model_name": "Whisper smail en - Song train",  #a 'pretty' name for your model (翻譯)給模型一個好名子
    "finetuned_from": useModel,
    "tasks": "automatic-speech-recognition",
}

### 上傳

In [ ]:
#儲存模型pipeline使用的資料
processor.save_pretrained("./" + "whisper-small-zh")
#上傳模型
trainer.push_to_hub(**kwargs)

### 訓練完後如何使用

In [ ]:
from transformers import pipeline
from YouTubeDownload import YouTubeDownload
from MusicTools import MusicTools
import os
import gradio as gr
modelStr = "small"#"small""medium"
if True:
    tempmodel = "Zzzkay1/whisper-small-zh"#r"H:\NFU\score\openai\whisper-medium"
else:
    tempmodel = "openai/whisper-" + modelStr 

from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq


pipe = pipeline(
    "automatic-speech-recognition",
    model=tempmodel,#改成你自己的模型tag
    return_timestamps=True,
)

def fun(url1):
    audio = YouTubeDownload.download_youtube_audio(url1)
    video = YouTubeDownload.download_youtube_video(url1)
    separtionMusic = MusicTools.run_separation(audio)
    merge = YouTubeDownload.merge_video_audio(video_path=video, audio_path=separtionMusic[0])

    #去掉多餘的引號&替換不合法字元
    merge = merge.strip('"').strip("'")
    safe_name = os.path.basename(merge).replace("／", "_").replace("/", "_").replace("\\", "_")
    
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, safe_name)

    if merge != out_path:
        try:
            os.replace(merge, out_path)  #搬移檔案
        except Exception as e:
            print("搬檔失敗:", e)
            out_path = merge  #如果搬失敗，就還是用原本路徑

    if not os.path.exists(out_path):
        return f"檔案不存在: {out_path}"

    results = pipe(separtionMusic[1])["text"]
    #return out_path  #給gr.Video()
    with open("OutputFile.txt","w",encoding="utf-8") as f:
        f.write(results if isinstance(results, str) else "\n".join(results))
    return results
"""
沒用
def transcribe(audio):
    text = pipe(audio)["text"]
    return text
"""

iface = gr.Interface(
    fn=fun,
    inputs=gr.Textbox(lines=1, placeholder="請輸入YouTube連結"), #輸入文字區
    outputs="text", #輸出為文字
    title="Whisper Small zh", #標題自訂
    description="Realtime demo for zh speech recognition using a fine-tuned Whisper small model.", #描述自訂
)
    
iface.launch() #啟動

In [ ]:
# 首先，請確認您已安裝 transformers, torch, gradio 等函式庫
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline 
import torch

# 假設這兩個模組已存在於您的專案中
from YouTubeDownload import YouTubeDownload
from MusicTools import MusicTools
import os
import gradio as gr

# --- 模型設定 ---
# 'large-v3' 是目前效果最好的開源模型之一
modelStr = "large-v3"
if True:
    tempmodel ="Zzzkay1/whisper-small-zh"#"Zzzkay1/whisper-medium-zh"#r"H:\NFU\score\openai\whisper-medium"
else:
    tempmodel = "openai/whisper-" + modelStr 

# --- 模型與處理器載入 ---
# 1. 設定裝置 (GPU 或 CPU)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"使用的裝置: {device}")

# 2. 載入 Processor 和 Model
print(f"正在載入 {tempmodel} 模型，這可能會需要一些時間...")
processor = AutoProcessor.from_pretrained(tempmodel)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    tempmodel, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
print("模型載入完成。")

# 3. 使用已載入的模型和處理器建立 pipeline
print("正在建立語音辨識 pipeline...")
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
    torch_dtype=torch_dtype,
)
print("Pipeline 建立完成。")


# --- 主要處理函式 (採用單一步驟) ---
def fun(url1):
    audio = YouTubeDownload.download_youtube_audio(url1)
    video = YouTubeDownload.download_youtube_video(url1)
    separtionMusic = MusicTools.run_separation(audio)
    merge = YouTubeDownload.merge_video_audio(video_path=video, audio_path=separtionMusic[0])

    #去掉多餘的引號&替換不合法字元
    merge = merge.strip('"').strip("'")
    safe_name = os.path.basename(merge).replace("／", "_").replace("/", "_").replace("\\", "_")
    
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, safe_name)

    if merge != out_path:
        try:
            os.replace(merge, out_path)  #搬移檔案
        except Exception as e:
            print("搬檔失敗:", e)
            out_path = merge  #如果搬失敗，就還是用原本路徑

    if not os.path.exists(out_path):
        return f"檔案不存在: {out_path}"

    vocal_audio_path = separtionMusic[1]
    
    print("\n--- 正在進行高精度辨識與時間碼對齊 (單步驟) ---")
    
    # 我們採用單一步驟來同時產生文字和時間戳。
    # `repetition_penalty` 對於防止模型在歌唱等困難音訊上陷入重複循環至關重要。
    # 雖然這可能會極其輕微地影響時間戳精度，但這是獲得正確、可用文字輸出的必要權衡。
    generate_kwargs = {
        "task": "transcribe",
        "language": "zh",
        "repetition_penalty": 1.4,
        "length_penalty": 1.1,
        "temperature": 0.0,
        "logprob_threshold": -0.7,
        "no_speech_threshold": 0.4,
        "num_beams": 5, 
    }

    # 同時啟用時間戳和抑制重複的參數
    results_dict = pipe(
        vocal_audio_path, 
        return_timestamps=True,
        generate_kwargs=generate_kwargs
    )
    
    print("辨識與對齊完成。")
    
    # --- 格式化最終輸出 ---
    chunks = results_dict.get("chunks", [])
    
    results = ""
    for chunk in chunks:
        if 'timestamp' in chunk and 'text' in chunk:
            start, end = chunk['timestamp']
            
            if start is not None and end is not None:
                text = chunk['text']
                
                start_time = f"{int(start // 3600):02d}:{int((start % 3600) // 60):02d}:{int(start % 60):02d}"
                end_time = f"{int(end // 3600):02d}:{int((end % 3600) // 60):02d}:{int(end % 60):02d}"
                
                results += f"[{start_time} -> {end_time}]{text}\n"

    if not results.strip():
        results = "未能辨識到任何有效的語音片段。"
    
    print(f"最終辨識結果:\n{results}")
    
    with open("OutputFile.txt","w",encoding="utf-8") as f:
        f.write(results)
    return results

# --- Gradio 介面 ---
iface = gr.Interface(
    fn=fun,
    inputs=gr.Textbox(lines=1, placeholder="請輸入YouTube連結"),
    outputs="text",
    title="Whisper Large-v3 (高精度歌詞辨識)",
    description="使用 Transformers Pipeline 進行高精度中文歌詞辨識。此版本採用單一步驟，能有效抑制重複並同時生成時間碼。",
)
    
iface.launch() #啟動



## 人聲分離顯存佔用最佳化測試

In [ ]:
import torch
import torchaudio
from torchaudio.pipelines import HDEMUCS_HIGH_MUSDB
from torchaudio.models import hdemucs_high
from psutil import virtual_memory
import subprocess
import os
import soundfile as sf
from torchaudio.transforms import Fade

def _separate_sources(model, mix, sample_rate, device, segment=15.0, overlap=0.1):
    """
    分段執行音源分離，避免一次丟整首造成爆顯存
    """
    batch, channels, length = mix.shape

    chunk_len = int(sample_rate * segment * (1 + overlap))
    start = 0
    end = chunk_len
    overlap_frames = int(overlap * sample_rate)
    fade = Fade(fade_in_len=0, fade_out_len=overlap_frames, fade_shape="linear")

    final = torch.zeros(batch, len(model.sources), channels, length, device=device)

    while start < length - overlap_frames:
        chunk = mix[:, :, start:end]
        with torch.no_grad():
            out = model.forward(chunk)
        out = fade(out)
        final[:, :, :, start:end] += out

        if start == 0:
            fade.fade_in_len = overlap_frames
            start += chunk_len - overlap_frames
        else:
            start += chunk_len
        end += chunk_len
        if end >= length:
            fade.fade_out_len = 0

    return final

def run_separation(audio_path, segment = 15.0, use_cuda = True):
    try:
        bundle = HDEMUCS_HIGH_MUSDB
        model = bundle.get_model()
    except Exception as e:
        return e

    try:
        GPUmem = torch.cuda.mem_get_info()[1] / 1024 / 1024 / 1024
        SYSmem = virtual_memory().total / 1024 / 1024 / 1024 / 2
        if torch.cuda.is_available() and GPUmem > 2 and use_cuda:
            device = torch.device("cuda")
        else:
            device = torch.device("cpu")
        model = model.to(device)

        input_file = audio_path.strip('"')
        if not os.path.exists(input_file):
            return f"找不到輸入檔案：{input_file}"

        ext = os.path.splitext(input_file)[1].lower()
        if ext == '.wav':
            waveform, sample_rate = torchaudio.load(input_file)
        else:
            temp_wav = "temp_convert.wav"
            try:
                subprocess.run([
                    "ffmpeg", "-y", "-i", input_file, "-acodec", "pcm_s16le", "-ar", "44100", temp_wav
                ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
                data, sample_rate = sf.read(temp_wav, dtype='float32')
                if data.ndim == 1:
                    data = data[:, None]
                waveform = torch.from_numpy(data.T)
            except subprocess.CalledProcessError as ffmpeg_err:
                return ffmpeg_err
            finally:
                if os.path.exists(temp_wav):
                    os.remove(temp_wav)

        if sample_rate != bundle.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=bundle.sample_rate)
            waveform = resampler(waveform)
            sample_rate = bundle.sample_rate

        waveform = waveform.to(device)

        #🔹 使用分段方式做分離（segment=15s, overlap=0.1）
        sources = _separate_sources(
            model=model,
            mix=waveform.unsqueeze(0),
            segment = segment,
            sample_rate=sample_rate,
            device=device
        )[0]

        base_name = os.path.splitext(os.path.basename(input_file))[0]
        output_dir = os.path.join(os.getcwd(), "score/output", base_name)
        os.makedirs(output_dir, exist_ok=True)

        other_mix = sources[0] + sources[1] + sources[2]
        vocals = sources[3]

        torchaudio.save(os.path.join(output_dir, base_name + '-vocals.wav'), vocals.cpu(), sample_rate)
        torchaudio.save(os.path.join(output_dir, base_name + '-other.wav'), other_mix.cpu(), sample_rate)

        print(f"已儲存：{os.path.join(output_dir, base_name + '-vocals.wav')}")
        print(f"已儲存：{os.path.join(output_dir, base_name + '-other.wav')}")

        del sources
        del waveform
        torch.cuda.empty_cache()

        output = [
            f"{os.path.join(output_dir, base_name + '-other.wav')}",
            f"{os.path.join(output_dir, base_name + '-vocals.wav')}"
        ]
        return output

    except Exception as e:
        try:
            if 'sources' in locals():
                del sources
            if 'waveform' in locals():
                del waveform
        except:
            pass
        torch.cuda.empty_cache()
        return e


In [ ]:
run_separation(r"H:\NFU\score\訓練模型區塊\download\Kimberley陳芳語《愛你AINI》Official MV(HD) - Kimberley Chen (youtube).mp3", segment=15, use_cuda=True)

## 網頁測試(暫定)

In [ ]:
import gradio as gr
import os
from MusicTools import MusicTools
from YouTubeDownload import YouTubeDownload

def fun(url1):
    audio = YouTubeDownload.download_youtube_audio(url1)
    video = YouTubeDownload.download_youtube_video(url1)
    separtionMusic = MusicTools.run_separation(audio)
    merge = YouTubeDownload.merge_video_audio(video_path=video, audio_path=separtionMusic[0])

    #去掉多餘的引號 & 替換不合法字元
    merge = merge.strip('"').strip("'")
    safe_name = os.path.basename(merge).replace("／", "_").replace("/", "_").replace("\\", "_")
    
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, safe_name)

    if merge != out_path:
        try:
            os.replace(merge, out_path)  #搬移檔案
        except Exception as e:
            print("搬檔失敗:", e)
            out_path = merge  #如果搬失敗，就還是用原本路徑

    if not os.path.exists(out_path):
        return f"檔案不存在: {out_path}"

    return out_path  #給 gr.Video()


iface = gr.Interface(
    fn=fun,
    inputs=gr.Textbox(lines=1, placeholder="請輸入YouTube連結"),
    outputs=gr.Video(),
    title="哈哈標題",
    description="輸入Youtube連結自動下載並分離人聲及音樂",
)

iface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


已儲存：h:\NFU\score\score/output\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)-vocals.wav
已儲存：h:\NFU\score\score/output\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)-other.wav


## 網頁測試(有時間碼的外部呼叫)注:無效

In [ ]:
import WebDisplay
web = WebDisplay()

In [ ]:
from MusicTools import MusicTools
MusicTools.run_ASR(r"H:\NFU\score\訓練模型區塊\周杰倫 Jay Chou【稻香 Rice Field】-Official Music Video-vocals.wav")

In [3]:
from YouTubeDownload import YouTubeDownload
YouTubeDownload.download_captions_by_language_code("https://youtu.be/H0rgSQXYd84?list=PL5keGpQTtuNLjK4_VV9YIQpxUYTCnzlI4",language_code="zh-TW")

'/download\\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)_zh-TW.srt'

In [ ]:
from MusicTools import MusicTools
MusicTools.run_ASR(r"H:\NFU\output\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)_merged.mp4")

選擇的檔案: H:\NFU\output\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)_merged.mp4
正在進行 VAD (語音活動偵測) 以過濾非人聲雜訊...
VAD 處理失敗，將使用原始檔案繼續: Error opening 'H:\\NFU\\output\\孫燕姿 Sun Yan-Zi -  逆光 Against The Light 4K MV (Official 4K UltraHD Video)_merged.mp4': Format not recognised.
使用的裝置: cuda:0
正在載入 Zzzkay1/whisper-small-zh 模型，這可能會需要一些時間...


Using cache found in C:\Users\a0987/.cache\torch\hub\snakers4_silero-vad_master
Device set to use cuda:0


開始辨識...
辨識過程發生錯誤: Soundfile is either not in the correct format or is malformed. Ensure that the soundfile has a valid audio file extension (e.g. wav, flac or mp3) and is not corrupted. If reading from a remote URL, ensure that the URL is the full address to **download** the audio file.


# 音樂升降調

In [9]:
from MusicTools import MusicTools
MusicTools.change_pitch_ffmpeg(r"H:\NFU\Featured_Demo\download\【我的少女時代 Our Times】Movie Theme Song - 田馥甄 Hebe Tien《小幸運 A Little Happiness》Official MV.m4a",-6)

'H:\\NFU\\Featured_Demo\\download\\【我的少女時代 Our Times】Movie Theme Song - 田馥甄 Hebe Tien《小幸運 A Little Happiness》Official MV_step-6.m4a'

# 本地模型讀取

In [1]:
import os
import torch
import torchaudio
import subprocess
import gc
import soundfile as sf
import tkinter as tk
from tkinter import filedialog
from psutil import virtual_memory
from torchaudio.pipelines import HDEMUCS_HIGH_MUSDB
from torchaudio.transforms import Fade

def run_ASR(audio="", Input_language="", model_dir=""):
    import torch
    import gc
    import os
    import math
    import soundfile as sf
    from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
    
    # 初始化 tkinter (用於檔案選擇視窗)
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
    except:
        pass 

    # --- 1. 選擇音訊檔案 ---
    if audio == "":
        current_dir = os.getcwd()
        file_path = filedialog.askopenfilename(
            title="請選擇音訊檔案",
            initialdir=current_dir,
            filetypes=[("音訊檔案", "*.wav *.mp3 *.m4a *.flac"), ("所有檔案", "*.*")]
        )
        if not file_path:
            print("未選擇檔案")
            return None
    else:
        file_path = audio.strip('"')

    file_path = os.path.abspath(file_path)
    audio_dir = os.path.dirname(file_path)
    filename = os.path.splitext(os.path.basename(file_path))[0]
    
    # 獲取音訊總長度
    try:
        info = sf.info(file_path)
        audio_duration = info.duration
    except:
        audio_duration = 999999.0

    # --- 2. 選擇/設定模型路徑 (修改處) ---
    # 如果呼叫函式時沒有給 model_dir，則跳出視窗讓使用者選擇
    target_model_path = model_dir
    
    if target_model_path == "":
        print("請選擇模型資料夾 (包含 config.json, model.safetensors 的資料夾)...")
        target_model_path = filedialog.askdirectory(
            title="請選擇 Whisper 模型資料夾",
            initialdir=os.getcwd()
        )

    # 如果使用者按取消或還是空的，使用預設線上模型
    if not target_model_path:
        print("未選擇本地模型，將使用預設 HuggingFace 線上模型。")
        target_model_path = "Zzzkay1/whisper-small-zh"
    else:
        print(f"將使用本地模型: {target_model_path}")

    # --- 3. 載入模型 ---
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    print(f"正在載入模型 {target_model_path}...")
    try:
        # local_files_only=True 可以強制只讀本地，但若路徑是資料夾，transformers 通常能自動識別
        processor = AutoProcessor.from_pretrained(target_model_path)
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            target_model_path, 
            torch_dtype=torch_dtype, 
            low_cpu_mem_usage=True, 
            use_safetensors=True
        )
        model.to(device)
    except Exception as e:
        print(f"模型載入失敗: {e}")
        print("請確認選擇的資料夾內是否包含 'config.json', 'preprocessor_config.json' 以及模型權重檔 (.safetensors 或 .bin)")
        return None

    # --- 4. VAD預處理與過濾 (保持原樣) ---
    print("正在進行VAD過濾...")
    processed_file_path = os.path.join(audio_dir, f"{filename}_vad_processed.wav")
    
    try:
        # 注意: Silero VAD 這裡還是從 torch.hub 讀取，若要本地化需要另外下載 repo
        vad_model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', force_reload=False, onnx=False)
        (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils
        
        # 讀取原始音訊
        wav = read_audio(file_path, sampling_rate=16000)
        
        # 取得原始的人聲時間戳
        raw_timestamps = get_speech_timestamps(
            wav, 
            vad_model, 
            threshold=0.4,           
            sampling_rate=16000, 
            min_speech_duration_ms=100,
            min_silence_duration_ms=100, 
            speech_pad_ms=50       
        )

        # 清洗時間戳
        clean_timestamps = []
        
        for stamp in raw_timestamps:
            start = stamp['start']
            end = stamp['end']
            duration_ms = (end - start) / 16000 * 1000
            
            if duration_ms < 400: 
                # print(f"發現短雜訊,已過濾: {duration_ms:.2f}ms")
                continue
            
            clean_timestamps.append(stamp)

        speech_timestamps = clean_timestamps

        if len(speech_timestamps) > 0:
            vad_audio = torch.zeros_like(wav)
            for stamp in speech_timestamps:
                vad_audio[stamp['start']:stamp['end']] = wav[stamp['start']:stamp['end']]
            
            save_audio(processed_file_path, vad_audio, sampling_rate=16000)
            print(f"VAD 過濾完成,暫存檔: {processed_file_path}")
            wav = vad_audio 
        else:
            print("警告: VAD 過濾後沒有剩餘人聲 (可能都是雜訊),將使用原始檔案。")

    except Exception as e:
        print(f"VAD 處理失敗,將使用原始檔案: {e}")

    # --- 5. 微切分與辨識 ---
    segments_for_output = []
    print(f"準備對 {len(speech_timestamps)} 個片段進行長度檢查與辨識...")

    MAX_CHUNK_DURATION = 25.0 

    final_chunks_to_process = []
    for stamp in speech_timestamps:
        start_sample = stamp['start']
        end_sample = stamp['end']
        duration_sec = (end_sample - start_sample) / 16000

        if duration_sec <= MAX_CHUNK_DURATION:
            final_chunks_to_process.append((start_sample, end_sample))
        else:
            num_sub_chunks = math.ceil(duration_sec / MAX_CHUNK_DURATION)
            chunk_samples = int(MAX_CHUNK_DURATION * 16000)
            for i in range(num_sub_chunks):
                sub_start = start_sample + (i * chunk_samples)
                sub_end = min(sub_start + chunk_samples, end_sample)
                if (sub_end - sub_start) / 16000 < 0.2: continue
                final_chunks_to_process.append((sub_start, sub_end))

    # 設定解碼參數
    lang = Input_language if Input_language and Input_language.lower() != "auto" else None
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=lang, task="transcribe")

    for idx, (start_sample, end_sample) in enumerate(final_chunks_to_process):
        
        vad_start_time = start_sample / 16000
        vad_end_time = end_sample / 16000
        
        segment_wav = wav[start_sample:end_sample].numpy()
        
        try:
            input_features = processor(
                segment_wav, 
                sampling_rate=16000, 
                return_tensors="pt"
            ).input_features.to(device).to(torch_dtype)

            if forced_decoder_ids is not None:
                    model.generation_config.forced_decoder_ids = forced_decoder_ids
                
                # 直接生成
            with torch.no_grad():
                generated_ids = model.generate(
                    input_features,
                    # forced_decoder_ids=forced_decoder_ids, # 這一行刪除或註解掉
                    max_new_tokens=255, 
                    temperature=0.0,
                    repetition_penalty=1.2
                    )
            """with torch.no_grad():
                generated_ids = model.generate(
                    input_features,
                    forced_decoder_ids=forced_decoder_ids,
                    max_new_tokens=255, 
                    temperature=0.0,
                    repetition_penalty=1.2
                )"""

            text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        except Exception as e:
            print(f"片段錯誤 ({vad_start_time:.1f}s): {e}")
            continue

        if len(text) < 1: continue
        if segments_for_output and text == segments_for_output[-1]['text']: continue

        segments_for_output.append({
            "start": vad_start_time,
            "end": vad_end_time,
            "text": text
        })

        if idx % 5 == 0:
            print(f"進度 {idx}/{len(final_chunks_to_process)}: {text[:10]}...")

    # --- 6. 輸出 SRT ---
    def format_time_srt(time_sec):
        hours = int(time_sec // 3600)
        minutes = int((time_sec % 3600) // 60)
        seconds = int(time_sec % 60)
        milliseconds = int((time_sec % 1) * 1000)
        return f"{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}"
        
    srt_path = os.path.join(audio_dir, f"{filename}.srt")
    with open(srt_path, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments_for_output, start=1):
            f.write(f"{i}\n")
            f.write(f"{format_time_srt(seg['start'])} --> {format_time_srt(seg['end'])}\n")
            f.write(f"{seg['text']}\n\n")
            
    txt_path = os.path.join(audio_dir, f"{filename}.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        for seg in segments_for_output:
            f.write(f"{seg['text']}\n")

    print(f"辨識完成,SRT 已輸出: {srt_path}")

    # --- 7. 清理 ---
    if os.path.exists(processed_file_path):
        try:
            # os.remove(processed_file_path) 
            pass
        except:
            pass

    del model
    del processor
    try:
        del vad_model
    except:
        pass
    gc.collect()
    torch.cuda.empty_cache()

    return [srt_path, "zh"]

In [4]:
from YouTubeDownload import YouTubeDownload
from MusicTools import MusicTools
audio = YouTubeDownload.download_youtube_audio("https://youtu.be/g0lNMzPC94k?si=r6lMi2K5A7dIifz-")
audio1 = MusicTools.run_separation(audio)
run_ASR(audio1[1],"")

已儲存：h:\NFU\Featured_Demo\score\score/output\炮仔聲 ~ 江蕙\炮仔聲 ~ 江蕙-vocals.wav
已儲存：h:\NFU\Featured_Demo\score\score/output\炮仔聲 ~ 江蕙\炮仔聲 ~ 江蕙-other.wav
請選擇模型資料夾 (包含 config.json, model.safetensors 的資料夾)...
將使用本地模型: H:/NFU/Featured_Demo/score/whisper-small-tw/checkpoint-2200
正在載入模型 H:/NFU/Featured_Demo/score/whisper-small-tw/checkpoint-2200...
正在進行VAD過濾...


Using cache found in C:\Users\a0987/.cache\torch\hub\snakers4_silero-vad_master


VAD 過濾完成,暫存檔: h:\NFU\Featured_Demo\score\score\output\炮仔聲 ~ 江蕙\炮仔聲 ~ 江蕙-vocals_vad_processed.wav
準備對 32 個片段進行長度檢查與辨識...
進度 0/32: 暗暗的蠟燭...
進度 5/32: 錯醉的傷手聽阮狗這段...
進度 10/32: 捌我聲找阮了來去叫一...
進度 15/32: 有聲就仔跤...
進度 20/32: 我按命知無行...
進度 25/32: 望我惜找咱著來去行嘸...
進度 30/32: 猶有...
辨識完成,SRT 已輸出: h:\NFU\Featured_Demo\score\score\output\炮仔聲 ~ 江蕙\炮仔聲 ~ 江蕙-vocals.srt


['h:\\NFU\\Featured_Demo\\score\\score\\output\\炮仔聲 ~ 江蕙\\炮仔聲 ~ 江蕙-vocals.srt',
 'zh']

# 中文微調

In [6]:
# ### 這裡開始


import pandas as pd
from datasets import load_dataset, Audio ,DatasetDict

#模型變數設置
#medium 
useModel = "openai/whisper-small"
language1 = "Chinese"#"English" #
language = "zh" #"en" #

#讀取原始CSV
df = pd.read_csv("processed_songs/metadata_all.csv")

df['file'] = "processed_songs/" + df['file'].astype(str)
#改欄位名稱符合Hugging Face標準
df = df.rename(columns={"file": "audio", "text": "sentence"})
df = df[["audio","sentence"]]

#存成新的CSV
df.to_csv("processed_songs/metadata_hf.csv", index=False)

#用Hugging Face載入新的CSV
dataset = load_dataset(
    "csv",
    data_files={"train": "processed_songs/metadata_hf.csv"}
)

#直接在cast_column中指定目標採樣率
#這會告訴datasets在讀取音檔時，自動解碼並重採樣到16000Hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
common_voice = dataset

#特徵提取器(預處理音檔)_選擇模型
#模型輸出後處理為文字格式的分詞器
#language要自己改
from transformers import WhisperFeatureExtractor, WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained(useModel)
tokenizer = WhisperTokenizer.from_pretrained(useModel, language=language1, task="transcribe")

input_str = common_voice["train"][0]["sentence"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


from transformers import WhisperProcessor, WhisperForConditionalGeneration, GenerationConfig
"""輸出格式範例(採樣率48K)
{'audio': {'path': '/home/sanchit_huggingface_co/.cache/huggingface/datasets/downloads/extracted/607848c7e74a89a3b5225c0fa5ffb9470e39b7f11112db614962076a847f3abf/cv-corpus-11.0-2022-09-21/hi/clips/common_voice_hi_25998259.mp3', 
           'array': array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 9.6724887e-07,
       1.5334779e-06, 1.0415988e-06], dtype=float32), 
           'sampling_rate': 48000},
 'sentence': 'खीर की मिठास पर गरमाई बिहार की सियासत, कुशवाहा ने दी सफाई'}
"""
processor = WhisperProcessor.from_pretrained(useModel, language=language, task="transcribe")

model = WhisperForConditionalGeneration.from_pretrained(useModel)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.decoder_start_token_id = 50258 
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

generation_config = GenerationConfig.from_model_config(model.config)
generation_config.update(
    language=language, 
    task="transcribe",
    forced_decoder_ids=None,
    max_length=225, # 配合你後面的設定
    suppress_tokens=[],
    begin_suppress_tokens=[220, 50257] # 抑制非文字符號
)
model.generation_config = generation_config

#print(common_voice["train"][0])


#from datasets import Audio
"""輸出格式範例(下採樣至16K)
{'audio': {'path': '/home/sanchit_huggingface_co/.cache/huggingface/datasets/downloads/extracted/607848c7e74a89a3b5225c0fa5ffb9470e39b7f11112db614962076a847f3abf/cv-corpus-11.0-2022-09-21/hi/clips/common_voice_hi_25998259.mp3', 
           'array': array([ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
       -3.4206650e-07,  3.2979898e-07,  1.0042874e-06], dtype=float32),
           'sampling_rate': 16000},
 'sentence': 'खीर की मिठास पर गरमाई बिहार की सियासत, कुशवाहा ने दी सफाई'}
"""
#common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

print(common_voice["train"][0])


#初始化processor(包含feature_extractor + tokenizer)
import torch
def prepare_dataset(batch, processor, torch):
    #processor = WhisperProcessor.from_pretrained(useModel, language="zh", task="transcribe")
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    batch["labels"] = torch.tensor(batch["labels"], dtype=torch.long)
    return batch

common_voice = common_voice.map(
    prepare_dataset,
    fn_kwargs={"processor":processor,"torch":torch},
    remove_columns=common_voice["train"].column_names,
    num_proc=14  #取決於你的CPU核心，自己填數字
)



#定義數據收集器
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. 處理音訊輸入
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2. 處理標籤 (Labels)
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 將 Padding 替換為 -100 (用於 Loss 計算)
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 如果標籤開頭已經是 BOS (50258)，切掉它 (避免重複)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        # --- 🔥 關鍵修正：手動生成 decoder_input_ids 🔥 ---
        # 我們不依賴模型自己產生，而是直接做給它
        # 邏輯：將 Labels 向右移一位，並在開頭補上 decoder_start_token_id (50258)
        
        decoder_input_ids = labels.new_zeros(labels.shape)
        decoder_input_ids[:, 1:] = labels[:, :-1].clone()
        decoder_input_ids[:, 0] = self.decoder_start_token_id
        
        # ⚠️ 非常重要：decoder 輸入不能有 -100，必須把 -100 還原成 pad_token_id
        # 因為輸入給解碼器的必須是真實存在的 Token ID
        pad_token_id = self.processor.tokenizer.pad_token_id
        decoder_input_ids.masked_fill_(decoder_input_ids == -100, pad_token_id)
        
        batch["decoder_input_ids"] = decoder_input_ids
        # --------------------------------------------------

        return batch

# 初始化數據整理器
# 確保你的 model.config.decoder_start_token_id 已經被設為 50258 (在上一段代碼中)
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=50258, # 強制指定 50258，確保萬無一失
)

import evaluate
import numpy as np

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    
    if pred_ids.ndim == 3:
        pred_ids = pred_ids.argmax(-1)

    # 統一使用 processor.tokenizer
    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # 根據全域變數 language 自動切換
    if language == "zh":
        # 中文：計算字元錯誤率 (CER)
        pred_str = [" ".join(list(s.replace(" ", ""))) for s in pred_str]
        label_str = [" ".join(list(s.replace(" ", ""))) for s in label_str]
    else:
        # 英文：轉小寫計算 WER
        pred_str = [s.lower() for s in pred_str]
        label_str = [s.lower() for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

#載入預訓練checkpoint
from transformers import WhisperForConditionalGeneration

#模型自己改


from transformers import Seq2SeqTrainingArguments,EarlyStoppingCallback

#如果不想將模型checkpoint上傳到Hub，設定push_to_hub=False
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-zh",
    per_device_train_batch_size=4,  #如果出現OOM(顯存超載)錯誤將此減半(初始值8)
    gradient_accumulation_steps=4,  #並將此乘2(初始值2)
    learning_rate=1e-5,
    warmup_steps=200,   # 縮短warmup，因為模型很快就收斂了
    max_steps=10000,
    gradient_checkpointing=False, #開啟這個可以節省顯存，雖然會慢一點
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    
    # --- 關鍵改動處 ---
    save_steps=100,            #每100步就檢查一次
    eval_steps=100,            #每100步就評估一次
    save_total_limit=3,        #只保留最好的兩個模型，省硬碟空間
    weight_decay=0.05,         #增加權重衰減，防止過擬合
    label_smoothing_factor=0.1, #增加標籤平滑，提高泛化能力
    # ----------------
    
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5 # 連續 5 次評估沒有改善則停止
)


#從訓練資料集中抓出測試資料
common_voice = common_voice["train"].train_test_split(test_size=0.1)


from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback],
)


#訓練模型
trainer.train()


#設定上傳模型數值
kwargs = {
    "dataset_tags": "Zzzkay1/song", #標籤，記得改
    "dataset": "Chinese songs * 25",  #a 'pretty' name for the training dataset (翻譯)給資料集設定一個你喜歡的名子
    "dataset_args": "config: zh, split: test",
    "language": "zh", #語言自己改
    "model_name": "Whisper smail zh - Song train",  #a 'pretty' name for your model (翻譯)給模型一個好名子
    "finetuned_from": useModel,
    "tasks": "automatic-speech-recognition",
}
# ### 上傳


#儲存模型pipeline使用的資料
processor.save_pretrained("./" + "whisper-small-en")
#上傳模型
#trainer.push_to_hub(**kwargs)

Generating train split: 0 examples [00:00, ? examples/s]

Input:                 我真的不明白昨天妳還很溫柔我大概犯了錯
Decoded w/ special:    <|startoftranscript|><|zh|><|transcribe|><|notimestamps|>我真的不明白昨天妳還很溫柔我大概犯了錯<|endoftext|>
Decoded w/out special: 我真的不明白昨天妳還很溫柔我大概犯了錯
Are equal:             True
{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x000001E823667E30>, 'sentence': '我真的不明白昨天妳還很溫柔我大概犯了錯'}


Map (num_proc=14):   0%|          | 0/1877 [00:00<?, ? examples/s]

C:\Users\a0987\AppData\Local\Temp\ipykernel_28872\1888855723.py:255: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss,Wer
100,2.687800,2.629751,51.798561
200,2.184600,2.325278,41.199041
300,1.830000,2.219373,29.928058
400,1.621200,2.209090,44.364508
500,1.553400,2.212729,27.146283
600,1.505800,2.208321,26.091127
700,1.487300,2.189849,26.187050
800,1.474800,2.195587,25.947242
900,1.467500,2.208885,25.851319
1000,1.460400,2.241190,24.988010


`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.Supp

[]

# 台語微調

In [2]:
import pandas as pd
from datasets import load_dataset, Audio, DatasetDict
import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperFeatureExtractor, 
    WhisperTokenizer, 
    WhisperProcessor, 
    WhisperForConditionalGeneration, 
    GenerationConfig,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    Seq2SeqTrainer
)

# ### 1. 變數與語言設置 ###
useModel = "openai/whisper-small"
language1 = "Chinese" 
language = "zh" 

# 設定輸出目錄名稱
output_model_dir = "./whisper-small-tw"

# ### 2. 資料處理 ###
# 讀取原始CSV
df = pd.read_csv("processed_songs/metadata_all.csv")

df['file'] = "processed_songs/" + df['file'].astype(str)
# 改欄位名稱符合Hugging Face標準
df = df.rename(columns={"file": "audio", "text": "sentence"})
df = df[["audio","sentence"]]

# 存成新的CSV
df.to_csv("processed_songs/metadata_hf.csv", index=False)

# 用Hugging Face載入新的CSV
dataset = load_dataset(
    "csv",
    data_files={"train": "processed_songs/metadata_hf.csv"}
)

# 強制重採樣到 16000Hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
common_voice = dataset

# ### 3. 特徵提取與分詞器 ###
feature_extractor = WhisperFeatureExtractor.from_pretrained(useModel)
tokenizer = WhisperTokenizer.from_pretrained(useModel, language=language1, task="transcribe")

# 初始化 Processor
processor = WhisperProcessor.from_pretrained(useModel, language=language, task="transcribe")

# ### 4. 資料預處理函數 ###
def prepare_dataset(batch, processor):
    audio = batch["audio"]
    # 提取音訊特徵 (Mel Spectrogram)
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    # 將文字轉為 Token IDs
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

# 套用預處理
common_voice = common_voice.map(
    prepare_dataset,
    fn_kwargs={"processor": processor},
    remove_columns=common_voice["train"].column_names,
    num_proc=14 # 請根據你的 CPU 核心數調整
)

# ### 5. 資料整理器 (Data Collator) - 包含 decoder_input_ids 修復 ###
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. 處理音訊輸入
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2. 處理標籤
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 將 Padding 替換為 -100
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 如果標籤開頭已經是 BOS (50258)，切掉它
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        # --- 手動生成 decoder_input_ids ---
        decoder_input_ids = labels.new_zeros(labels.shape)
        decoder_input_ids[:, 1:] = labels[:, :-1].clone()
        decoder_input_ids[:, 0] = self.decoder_start_token_id
        
        pad_token_id = self.processor.tokenizer.pad_token_id
        decoder_input_ids.masked_fill_(decoder_input_ids == -100, pad_token_id)
        
        batch["decoder_input_ids"] = decoder_input_ids
        # --------------------------------

        return batch

# 初始化 Data Collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=50258, 
)

# ### 6. 評估指標 (WER/CER) ###
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    
    if pred_ids.ndim == 3:
        pred_ids = pred_ids.argmax(-1)

    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    if language == "zh":
        pred_str = [" ".join(list(s.replace(" ", ""))) for s in pred_str]
        label_str = [" ".join(list(s.replace(" ", ""))) for s in label_str]
    else:
        pred_str = [s.lower() for s in pred_str]
        label_str = [s.lower() for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# ### 7. 模型設置 ###
model = WhisperForConditionalGeneration.from_pretrained(useModel)

# 強制關閉 use_cache
model.config.use_cache = False 

# 設定生成組態
model.config.forced_decoder_ids = None 
model.config.suppress_tokens = []
model.config.decoder_start_token_id = 50258 
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

generation_config = GenerationConfig.from_model_config(model.config)
generation_config.update(
    language=language, 
    task="transcribe",
    forced_decoder_ids=None,
    max_length=225,
    use_cache=False 
)
model.generation_config = generation_config

# 切分測試集
common_voice = common_voice["train"].train_test_split(test_size=0.1)

# ### 8. 訓練參數 (重點修改處) ###
training_args = Seq2SeqTrainingArguments(
    output_dir=output_model_dir,
    
    # --- 防止 OOM 與 報錯 的設定 ---
    per_device_train_batch_size=2,   # 降到 2 (如果顯存還夠可試試 4，不夠改 1)
    gradient_accumulation_steps=8,   # 配合上面降 batch size，這裡乘 2 倍 (原本4->8)
    gradient_checkpointing=False,    # 🚨 關鍵：關閉它，解決 RuntimeError
    # -----------------------------
    
    learning_rate=1e-5,
    warmup_steps=200, 
    max_steps=10000,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=4,    # 評估時也稍微降低一點比較安全
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,            
    eval_steps=100,            
    save_total_limit=3,        
    weight_decay=0.05,        
    label_smoothing_factor=0.1, 
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5 
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.tokenizer, # 修正：這裡應該傳入 tokenizer，不是 feature_extractor
    callbacks=[early_stopping_callback],
)

# ### 9. 開始訓練 ###
trainer.train()

# ### 10. 上傳與儲存設定 ###
kwargs = {
    "dataset_tags": "Zzzkay1/song",
    "dataset": "Taiwanese Songs (Min Nan)", 
    "dataset_args": "config: zh, split: test",
    "language": "nan",
    "model_name": "Whisper Small Taiwanese (Min Nan)", 
    "finetuned_from": useModel,
    "tasks": "automatic-speech-recognition",
    "tags": ["whisper", "speech-recognition", "zh-TW", "nan", "Taiwanese"]
}

# 儲存模型
processor.save_pretrained(output_model_dir)
trainer.save_model(output_model_dir)

# 上傳 (解開註解使用)
# trainer.push_to_hub(**kwargs)

Generating train split: 0 examples [00:00, ? examples/s]

Map (num_proc=14):   0%|          | 0/758 [00:00<?, ? examples/s]

C:\Users\a0987\AppData\Local\Temp\ipykernel_38156\3096806193.py:205: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss,Wer
100,3.576700,3.500457,78.653113
200,1.971900,2.634451,59.339263
300,1.593500,2.474607,53.875476
400,1.519700,2.506385,32.655654
500,1.488400,2.474892,31.766201
600,1.470500,2.468593,31.639136
700,1.460700,2.470165,30.241423
800,1.451500,2.520490,29.479034
900,1.447900,2.421546,28.081321
1000,1.443800,2.507450,28.081321


`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.Supp

# 資料集時長統計

In [8]:
import os
from mutagen import File
from mutagen.wave import WAVE
from mutagen.mp3 import MP3

def calculate_dataset_duration(folder_path):
    """
    計算指定資料夾及其子資料夾內所有 wav 和 mp3 檔案的總時長。
    """
    total_seconds = 0
    file_count = 0
    
    # 獲取資料集名稱（從路徑中提取最後一個資料夾名）
    dataset_name = os.path.basename(os.path.normpath(folder_path))
    
    print(f"正在分析資料集: {dataset_name}，請稍候...")

    # os.walk 會遞迴遍歷所有子資料夾
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.wav', '.mp3')):
                file_path = os.path.join(root, file)
                try:
                    # 使用 mutagen 讀取音訊檔案
                    audio = File(file_path)
                    
                    if audio is not None and audio.info is not None:
                        total_seconds += audio.info.length
                        file_count += 1
                except Exception as e:
                    print(f"無法讀取檔案 {file}: {e}")

    if file_count == 0:
        print("未找到任何 .wav 或 .mp3 檔案。")
        return

    # 時間轉換
    total_minutes = total_seconds / 60
    total_hours = total_seconds / 3600

    # 格式化輸出
    # 保留小數點後兩位
    print("-" * 30)
    print(f"統計結果:")
    print(f"掃描檔案數: {file_count} 個")
    print(f"{dataset_name}:總時長{int(total_minutes)}分鐘({total_hours:.2f}小時)")
    print("-" * 30)

if __name__ == "__main__":
    # --- 請在這裡輸入你的資料夾路徑 ---
    # Windows 路徑範例: r"D:\Datasets\MyVoiceData"
    # Mac/Linux 路徑範例: "/home/user/datasets/voice_data"
    
    target_path = r"H:\NFU\Featured_Demo\source\processed_songs_台"#input("請輸入資料夾路徑: ").strip('"') # 移除可能因拖曳產生的引號
    
    if os.path.exists(target_path):
        calculate_dataset_duration(target_path)
    else:
        print("錯誤：找不到指定的路徑，請確認後再試。")

正在分析資料集: processed_songs_台，請稍候...
------------------------------
統計結果:
掃描檔案數: 858 個
processed_songs_台:總時長84分鐘(1.40小時)
------------------------------
